# Archived analysis workflow

This notebook records the author workflow used during revision. It requires the corresponding external model-ready inputs, saved predictions and/or checkpoints. Its `/content/drive/MyDrive` paths describe the author environment and must be reviewed before execution. Do not run all cells merely to inspect the repository. See [analysis/README.md](../README.md) and the main repository README for the supported benchmark entry point.


# SpaGAT R3.5 / R3.7：核对模型并导出 gene recovery 与 signed masking

**运行方式：Colab → 更改运行时类型 → GPU（例如 T4），然后从上到下运行。建议新建运行时。无需手动填路径，无需安装额外包。**

本次是已有 checkpoint 的推理与导出，不训练。程序先检查两个指定候选：validation indices、旧 activations 中的细胞标签/坐标、19,659/6,892 个 eligible receivers，以及旧结果中对应的 full MSE。只有一个候选匹配才继续；不匹配或两个都匹配时停止并生成诊断 ZIP，不自动选分数更高或更新的文件。

模型来源是你上传的 locator ZIP 中的源码。严格加载权重；数据分块缓存到运行时磁盘，避免将全部基因矩阵和逐细胞 baseline 同时堆入内存。输入和旧结果不覆盖。代码已做语法和模拟数据检查，但尚未在你的真实 checkpoint/GPU 上验证。

**范围：本 notebook 完成逐基因预测恢复和原等数量（matched-count）masking 的 signed export。它不重跑 fraction50 控制，也不把 latent channels 当 biological programs。**

后续 program recovery 可以使用此次保存的预测，不需要重新运行模型。所有结果来自原 validation setting，不是 Liver independent test。目标直接沿用原 loader 中的 processed gene values，不再自行减一次 baseline；输出后仍需核对这些输入的 residual 预处理语义。


## 1. 挂载 Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. 准备与你上传版本一致的模型代码（只写运行时目录）

In [ ]:
from pathlib import Path
import sys, json, hashlib
FROZEN_SOURCE = {'attention.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom .embedding import FFN\n\ndef rho(x):\n    return torch.sqrt(F.relu(x)) - torch.sqrt(F.relu(-x))\n\ndef get_diagonal(x):\n    index = torch.arange(0, x.shape[1], 1)\n    return x[:, index, index, :]\n\n\ndef _normalize_pairwise_tensors(node, edge, embedding, edge_dim=None):\n    """Normalize Embedding\'s initial singleton receiver axis.\n\n    ``Embedding.forward`` supplies pairwise tensors as ``(B, 1, N, E)``: one\n    sender-wise edge representation shared by every receiver.  GRIT attention\n    operates on an explicit receiver/sender representation ``(B, N, N, E)``.\n    Expanding the singleton receiver axis before Q/K--edge arithmetic prevents\n    PyTorch from broadcasting the batch dimension into an unintended\n    ``(B, B, N, N, E)`` tensor.  Later GRIT layers already use the expanded\n    four-dimensional convention.\n    """\n    if node.ndim != 3:\n        raise ValueError(f"node must have shape (B, N, node_dim); got {tuple(node.shape)}.")\n    B, N, _ = node.shape\n\n    def normalize(name, tensor):\n        if tensor.ndim != 4:\n            raise ValueError(\n                f"{name} must have shape (B, 1, N, edge_dim) or (B, N, N, edge_dim); "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[0] != B or tensor.shape[2] != N:\n            raise ValueError(\n                f"{name} batch and sender dimensions must be B={B} and N={N}; "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[1] == 1:\n            tensor = tensor.expand(B, N, N, tensor.shape[-1])\n        elif tensor.shape[1] != N:\n            raise ValueError(\n                f"{name} receiver dimension must be 1 or N={N}; got {tuple(tensor.shape)}."\n            )\n        return tensor\n\n    edge = normalize("edge", edge)\n    embedding = normalize("embedding", embedding)\n    if edge.shape[-1] != embedding.shape[-1]:\n        raise ValueError(\n            "edge and embedding must have the same feature dimension; got "\n            f"{edge.shape[-1]} and {embedding.shape[-1]}."\n        )\n    if edge_dim is not None and edge.shape[-1] != edge_dim:\n        raise ValueError(\n            f"edge and embedding feature dimension must be edge_dim={edge_dim}; "\n            f"got {edge.shape[-1]}."\n        )\n    return edge, embedding\n\nclass GRIT_attention(nn.Module):\n    def __init__(self, node_dim, edge_dim, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        if routing_mode not in {"grit", "program"}:\n            raise ValueError("routing_mode must be \'grit\' or \'program\'.")\n        self.routing_mode = routing_mode\n        self.node_dim = node_dim\n        self.base_edge_dim = edge_dim\n        self.edge_dim = edge_dim * 2\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_V = nn.Linear(node_dim, node_dim)\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_En = nn.Linear(edge_dim, node_dim)\n        self.W_A = nn.Linear(edge_dim, 1, bias=False)\n        self.W_No = nn.Linear(node_dim, node_dim)\n        self.W_Eo = nn.Linear(edge_dim, edge_dim, bias=False)\n\n        # Program tokens live in the common edge-feature space (edge_dim).  They\n        # are supplied by a parent model so that token k has the same biological\n        # identity in every graph layer; these projections remain layer-specific.\n        if self.program_aware:\n            self.W_program_edge = nn.Linear(edge_dim, edge_dim, bias=False)\n            self.W_program_receiver = nn.Linear(node_dim, edge_dim, bias=False)\n            self.W_program_gate = nn.Linear(node_dim, edge_dim, bias=False)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Run edge-aware attention, optionally routed through communication programs.\n\n        Args:\n            x: ``[node, edge, embedding]`` with node ``(B, N, node_dim)`` and\n                broadcastable edge/embedding tensors ``(B, 1 or N, N, edge_dim)``.\n            program_tokens: Optional shared token bank ``(K, edge_dim)``.  A\n                token is a globally named latent communication program; the\n                same bank should be passed to every program-aware GRIT layer.\n            program_edge_mask: Optional inference-only ``(B, N, N, K)`` mask.\n                Zero entries remove sender edges from a program\'s softmax.\n\n        Returns:\n            ``[node_updated, edge_updated, program_state]``.  ``program_state``\n            is ``None`` in baseline mode, preserving the original computation.\n        """\n        node, edge, embedding = x\n        B, N, C = node.shape\n\n        # Always normalize the initial Embedding output, including the legacy\n        # path.  The routing mode must not determine whether Q/K arithmetic is\n        # performed on a correctly shaped pairwise edge tensor.\n        edge, embedding = _normalize_pairwise_tensors(\n            node, edge, embedding, edge_dim=self.base_edge_dim\n        )\n\n        if program_tokens is not None and not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_attention was "\n                "constructed with program_aware=False."\n            )\n        if program_tokens is not None and program_tokens.ndim != 2:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim); got "\n                f"{tuple(program_tokens.shape)}."\n            )\n        if program_tokens is not None and program_tokens.shape[1] != self.base_edge_dim:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim), where "\n                f"edge_dim={self.base_edge_dim}; got {tuple(program_tokens.shape)}."\n            )\n        if program_edge_mask is not None and program_tokens is None:\n            raise ValueError("program_edge_mask requires program-aware routing tokens.")\n\n        Q = self.W_Q(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 1, 2)\n        K = self.W_K(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 2, 1)\n        QK = (Q @ K).permute(0, 2, 3, 1)\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding)\n        V = self.W_V(node)\n\n        if program_tokens is None:\n            # This is the original GRIT message-passing path, kept numerically\n            # unchanged when no shared program bank is supplied.\n            alpha = self.W_A(edge).squeeze(dim=-1)\n            alpha = F.softmax(alpha, dim=-1)\n            node = alpha @ V + self.W_En(torch.sum(edge * alpha.unsqueeze(dim=-1), dim=-2))\n            node = self.W_No(node)\n            edge = self.W_Eo(edge)\n            return [node, edge, None]\n\n        if self.routing_mode == "grit":\n            if program_tokens.shape[0] != 1:\n                raise ValueError("routing_mode=\'grit\' is defined for the controlled K=1 ablation only.")\n            # Controlled routing ablation: retain GRIT\'s original scalar edge\n            # routing and expose it as the K=1 program-state interface.  The\n            # downstream program gate/messages/free decoder are unchanged.\n            program_attention = F.softmax(self.W_A(edge).squeeze(dim=-1), dim=-1).unsqueeze(dim=-1)\n        else:\n            # Program routing: the edge, receiver, and shared program token\n            # interact multiplicatively. This is the pre-existing SpaGP score.\n            edge_program = self.W_program_edge(edge)\n            receiver_program = self.W_program_receiver(node)\n            program_scores = torch.einsum(\n                "bije,bie,ke->bijk", edge_program, receiver_program, program_tokens\n            ) / (self.base_edge_dim ** 0.5)\n            program_attention = None\n        if program_edge_mask is not None:\n            expected_shape = (B, N, N, program_tokens.shape[0])\n            if tuple(program_edge_mask.shape) != expected_shape:\n                raise ValueError(\n                    f"program_edge_mask must have shape {expected_shape}; got "\n                    f"{tuple(program_edge_mask.shape)}."\n                )\n            allowed = program_edge_mask.to(device=program_scores.device, dtype=torch.bool)\n            if not torch.all(allowed.any(dim=2)):\n                raise ValueError("program_edge_mask removes every sender for at least one receiver/program.")\n            # This is inference-time graph intervention: masked edges receive\n            # exactly zero attention and remaining edges are renormalized by\n            # the existing neighbor softmax.\n            if self.routing_mode == "grit":\n                grit_scores = self.W_A(edge).squeeze(dim=-1).masked_fill(~allowed[..., 0], float("-inf"))\n                program_attention = F.softmax(grit_scores, dim=-1).unsqueeze(dim=-1)\n            else:\n                program_scores = program_scores.masked_fill(~allowed, float("-inf"))\n                program_attention = F.softmax(program_scores, dim=2)\n        elif program_attention is None:\n            program_attention = F.softmax(program_scores, dim=2)\n\n        # Aggregate node and edge evidence independently for each program:\n        # m[i, k] = sum_j alpha[i, j, k] * message[i, j].\n        node_messages = torch.einsum("bijk,bjd->bikd", program_attention, V)\n        edge_messages = torch.einsum("bijk,bije->bike", program_attention, edge)\n        program_messages = node_messages + self.W_En(edge_messages)\n\n        # Receiver-specific, non-negative gates fuse the K program messages back\n        # into the original node_dim, keeping the downstream tensor shape intact.\n        gate_scores = torch.einsum(\n            "bin,kn->bik", self.W_program_gate(node), program_tokens\n        ) / (self.base_edge_dim ** 0.5)\n        program_activations = F.softplus(gate_scores)\n        node = torch.sum(program_messages * program_activations.unsqueeze(dim=-1), dim=2)\n        node = self.W_No(node)\n        edge = self.W_Eo(edge)\n\n        program_state = {\n            "attention": program_attention,\n            "messages": program_messages,\n            "activations": program_activations,\n        }\n        return [node, edge, program_state]\n\nclass DegScaler(nn.Module):\n    def __init__(self, node_dim):\n        super().__init__()\n        scaler = (2 / node_dim) ** 0.5\n        self.theta1 = nn.Parameter(torch.randn(node_dim) * scaler)\n        self.theta2 = nn.Parameter(torch.randn(node_dim) * scaler)\n\n    def forward(self, node, degree):\n        node = self.theta1 * node + torch.log(degree + 1).unsqueeze(dim=-1) * node * self.theta2\n        return node\n\nclass Multi_Head_Attention(nn.Module):\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8):\n        super().__init__()\n        self.attentions = nn.ModuleList([\n            GRIT_attention(node_dim, edge_dim, att_dim) for _ in range(num_heads)\n        ])\n        self.W_hn = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\n        self.W_he = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\n\n    def forward(self, x):\n        results = [attentioni(x) for attentioni in self.attentions]\n        node = (torch.stack([tmp[0] for tmp in results], dim=-1) @ self.W_hn).squeeze(dim=-1)\n        edge = torch.stack([tmp[1] for tmp in results], dim=-1) @ self.W_he\n        edge = edge.squeeze(dim=-1)\n        return [node, edge, x[2]]\n\nclass GRIT_encoder(nn.Module):\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        self.edge_dim = edge_dim\n        self.routing_mode = routing_mode\n        self.attentions = Multi_Head_Attention(node_dim, edge_dim, num_heads, att_dim)\n        if program_aware:\n            # Keep Multi_Head_Attention unchanged for baseline callers, but replace\n            # its heads with token-aware GRIT attention modules for this encoder.\n            # The shared tokens themselves are supplied to forward(), not owned by\n            # a layer, so program index k remains consistent across GNN layers.\n            self.attentions.attentions = nn.ModuleList([\n                GRIT_attention(node_dim, edge_dim, att_dim, program_aware=True, routing_mode=routing_mode)\n                for _ in range(num_heads)\n            ])\n        self.FFN = FFN(node_dim)\n        self.ln1 = nn.LayerNorm(node_dim)\n        self.ln2 = nn.LayerNorm(node_dim)\n        self.ln_edge = nn.LayerNorm(edge_dim)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Encode one graph layer, optionally with shared program-aware routing.\n\n        Without ``program_tokens``, this follows the original GRIT encoder path\n        and returns ``[node, edge, distance_embedding]``.  With a shared token\n        bank, each head returns program-specific routing state; the encoder\n        combines it across heads and returns it as a fourth element.\n        """\n        node, edge, distance = x\n        # Keep the residual edge tensor in the same four-dimensional convention\n        # as GRIT_attention.  This is essential for the baseline residual add.\n        edge, distance = _normalize_pairwise_tensors(\n            node, edge, distance, edge_dim=self.edge_dim\n        )\n        x = [node, edge, distance]\n        if program_tokens is None:\n            # Preserve the existing multi-head aggregation and output contract.\n            x = self.attentions(x)\n            edge = self.ln_edge(edge + x[1])\n            node = self.ln1(node + x[0])\n            node = self.ln2(node + self.FFN(node))\n            return [node, edge, distance]\n\n        if not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_encoder was "\n                "constructed with program_aware=False."\n            )\n\n        # Multi_Head_Attention\'s baseline forward intentionally remains unchanged.\n        # Here, route the same shared program tokens through each token-aware head\n        # and use its existing learned head weights for node and edge aggregation.\n        results = [\n            attentioni(x, program_tokens, program_edge_mask=program_edge_mask)\n            for attentioni in self.attentions.attentions\n        ]\n        node_attention = (\n            torch.stack([result[0] for result in results], dim=-1) @ self.attentions.W_hn\n        ).squeeze(dim=-1)\n        edge_attention = torch.stack([result[1] for result in results], dim=-1) @ self.attentions.W_he\n        edge_attention = edge_attention.squeeze(dim=-1)\n\n        # Combine per-head routing metadata with normalized head weights.  This\n        # yields one program axis K shared by all heads and preserves alpha\'s\n        # interpretation as a neighbor distribution for each receiver/program.\n        head_weights = F.softmax(self.attentions.W_hn.squeeze(dim=-1), dim=0)\n        program_state = {\n            key: torch.sum(\n                torch.stack([result[2][key] for result in results], dim=-1)\n                * head_weights,\n                dim=-1,\n            )\n            for key in ("attention", "messages", "activations")\n        }\n\n        # Pairwise tensors were normalized at entry, so the residual update is\n        # always performed in the explicit ``(B, N, N, E)`` convention.\n        edge = self.ln_edge(edge + edge_attention)\n        node = self.ln1(node + node_attention)\n        node = self.ln2(node + self.FFN(node))\n        return [node, edge, distance, program_state]\n\nclass GRIT_encoder_last_layer(nn.Module):\n    def __init__(self, node_dim, in_node, edge_dim, node_dim_small=16, att_dim=8):\n        super().__init__()\n        self.node_dim_small = node_dim_small\n        self.in_node = in_node\n        self.edge_dim = edge_dim * 2\n        self.att_dim = att_dim\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_En = nn.Linear(edge_dim, node_dim)\n        self.W_A = nn.Linear(edge_dim, in_node, bias=False)\n        self.node_transform = nn.Linear(node_dim, in_node)\n        self.edge_transform = nn.Linear(edge_dim, in_node)\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\n        self.head2 = nn.Sequential(nn.LayerNorm(in_node), FFN(in_node))\n        self.scaler = 2 / (node_dim ** 0.5)\n\n    def forward(self, x):\n        node, edge, embedding = x\n        B, N, C = node.shape\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\n        K = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\n        QK = (Q @ K).permute(0, 2, 3, 1)\n        edge = edge[:, 0:1, :, :]\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding[:, 0:1, :, :])\n        edge = edge[:, 0, 1:, :]\n        alphas = F.softmax(self.W_A(edge).permute(0, 2, 1), dim=-1).permute(0, 2, 1)\n        edge = self.edge_transform(edge) * alphas\n        node = self.node_transform(node[:, 1:, :]) * alphas\n        tmp = self.head(torch.concat([edge, node], dim=-1))\n        node = torch.sum(tmp, dim=-2) * self.scaler\n        return [node, [tmp.permute(0, 2, 1).unsqueeze(dim=-2), edge.permute(0, 2, 1).unsqueeze(dim=-2)]]\n\nclass ProgramLastLayer(nn.Module):\n    def __init__(self, node_dim, in_node, edge_dim, n_programs=8, node_dim_small=16, att_dim=8):\n        super().__init__()\n        self.node_dim_small = node_dim_small\n        self.in_node = in_node\n        self.n_programs = n_programs\n        self.edge_dim = edge_dim * 2\n        self.att_dim = att_dim\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_A = nn.Linear(edge_dim, in_node * n_programs, bias=False)\n        self.node_transform = nn.Linear(node_dim, in_node)\n        self.edge_transform = nn.Linear(edge_dim, in_node)\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\n        self.scaler = 2 / (node_dim ** 0.5)\n\n    def forward(self, x):\n        node, edge, embedding = x\n        B, N, C = node.shape\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\n        K_mat = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\n        QK = (Q @ K_mat).permute(0, 2, 3, 1)\n        edge = edge[:, 0:1, :, :]\n        edge = F.gelu(\n            rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge))\n            + self.W_Eb(edge)\n            + QK[:, :, :, self.edge_dim // 2:]\n            + embedding[:, 0:1, :, :]\n        )\n        edge = edge[:, 0, 1:, :]\n        alphas_all = self.W_A(edge)\n        alphas_all = alphas_all.reshape(B, N - 1, self.in_node, self.n_programs)\n        alphas_all = alphas_all.permute(0, 3, 2, 1)\n        alphas_all = F.softmax(alphas_all, dim=-1)\n        alphas_all = alphas_all.permute(0, 3, 1, 2)\n        edge_feat = self.edge_transform(edge)\n        node_feat = self.node_transform(node[:, 1:, :])\n        program_outputs = []\n        program_influences = []\n        for k in range(self.n_programs):\n            alpha_k = alphas_all[:, :, k, :]\n            edge_k = edge_feat * alpha_k\n            node_k = node_feat * alpha_k\n            tmp_k = self.head(torch.cat([edge_k, node_k], dim=-1))\n            output_k = torch.sum(tmp_k, dim=1) * self.scaler\n            program_outputs.append(output_k)\n            program_influences.append(tmp_k)\n        y_pred = torch.stack(program_outputs, dim=0).sum(dim=0)\n        total_influence = torch.stack(program_influences, dim=0).sum(dim=0)\n        influence_for_compat = total_influence.permute(0, 2, 1).unsqueeze(dim=-2)\n        program_info = {\n            \'per_program_influence\': torch.stack(program_influences, dim=0),\n            \'per_program_output\': torch.stack(program_outputs, dim=0),\n            \'per_program_attention\': alphas_all,\n        }\n        return [y_pred, [influence_for_compat, program_info]]\n', 'preprocess.py': 'import torch\n\ndef process(x,judge_distance=28):\n    \'\'\'\n    :param x:\n        "x": exp,\n        "y": y,\n        "ligand":ligand,\n        "receptor":receptor,\n        "cell_types": cell_types,\n        "morphology": volumes,\n        "position_x": centerx,\n        "position_y": centery\n    :return:\n        "x": b,n,c1\n        "interactions": b,n,n,c2\n        "mask": b,n,n\n        "cell_types": b,n\n        "distance_matrix": b,n,n,c3\n        "y": b,c\n        "morphology": b,n,c4\n    \'\'\'\n    dx = x["position_x"][:, 0:1] - x["position_x"]\n    dy = x["position_y"][:, 0:1] - x["position_y"]\n    distances=torch.sqrt(torch.square(dx)+torch.square(dy))\n\n    \'\'\'mask=torch.where(distances<judge_distance,\n                     torch.ones_like(distances,device=distances.device),\n                     torch.zeros_like(distances,device=distances.device))\'\'\'\n    distance_matrix = torch.stack([1 / (distances + 1), 1 / (torch.sqrt(distances) + 1),\n                                   1 / (1 + torch.square(distances)), torch.exp(-distances),\n                                   torch.exp(-torch.square(distances))], dim=-1)\n    return {\n        "x":x["x"],\n        "type_exp": x["type_exp"],\n        "y":x["y"],\n        "cell_types": x["cell_types"],\n        "distance_matrix":distance_matrix\n    }', 'embedding.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom . import preprocess\n\nclass FFN(nn.Module):\n    def __init__(self,dim,in_dim=None,out_dim=None,bias=True):\n        super().__init__()\n        if in_dim is None:\n            in_dim=dim\n        if out_dim is None:\n            out_dim=dim\n\n        self.linear1=nn.Linear(in_dim,dim*4,bias=bias)\n        self.linear2=nn.Linear(dim*4,out_dim,bias=bias)\n        self.act=nn.GELU()\n\n    def forward(self,x):\n        return self.linear2(self.act(self.linear1(x)))\n\ndef group_strings_by_numbers(strings, numbers):\n    """\n    Group a list of strings based on unique numbers and return as two ordered lists.\n\n    :param strings: List of strings to be grouped.\n    :param numbers: List of numbers indicating the grouping.\n    :return: Two lists, one with unique numbers and the other with corresponding grouped strings.\n    """\n    numbers=numbers.copy()[:len(strings)]\n    unique_numbers = sorted(set(numbers))\n    grouped_strings = [[] for _ in unique_numbers]\n\n    for string, number in zip(strings, numbers):\n        index = unique_numbers.index(number)\n        grouped_strings[index].append(string)\n\n    return unique_numbers, grouped_strings\n\nclass Embedding(nn.Module):\n    def __init__(self,genes,ligands_info,node_dim,edge_dim,use_cell_type_embedding=True):\n        super().__init__()\n        self.use_cell_type_embedding=use_cell_type_embedding\n        self.ligands_info=ligands_info\n\n        # for scaling the data\n        #self.exp_scaler=nn.Parameter(torch.ones((len(genes))))\n\n        # for node\n        self.cell_encoder1 = FFN(node_dim, in_dim=len(genes))\n        self.cell_encoder2 = nn.Embedding(29,node_dim)\n\n        # for distance\n        self.distance_scaler=nn.Sequential(FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5),nn.Sigmoid())\n        self.distance_encoder=FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5)\n        self.distance_embedding = nn.Sequential(FFN(edge_dim, in_dim=5), nn.Sigmoid())\n\n        # for edge\n        self.ligands_encoder=nn.Linear(len(ligands_info[0]),edge_dim)\n        self.edge_encoder=nn.Linear(len(ligands_info[0])+edge_dim,edge_dim,bias=False)\n        self.scaler=edge_dim**0.5\n\n        # calculate ligand scores\n        self.ligands_index=[]\n        for i in range(len(self.ligands_info[0])):\n            ligandi = self.ligands_info[0][i]\n            ligand_groupi = self.ligands_info[1][i]\n            uniquei, groupi = group_strings_by_numbers(ligandi, ligand_groupi)\n            groupi=[torch.LongTensor([genes.index(groupi[j][k]) for k in range(len(groupi[j]))]) for j in range(len(groupi))]\n            self.ligands_index.append(groupi)\n\n    def forward(self, x, ligand_feature_mask=None):\n        \'\'\'\n        :param x:\n        x["x"]--b,n,in_node\n        x["distance_matrix"]--b,5,n,n\n        x["cell_types"]--long,b,n\n        x["type_exp"]--b,n,in_node\n        :return:\n        node_features:b,n,node_dim\n        edge1:b,n,n,edge_dim\n        edge2:b,n,n,edge_dim+5\n        rw: b,n,n,num_hops+5\n        \'\'\'\n        x=preprocess.process(x)\n\n        # node features\n        #scaler=torch.square(self.exp_scaler)\n        exp=x["type_exp"]+x["x"]\n        exp=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),x["type_exp"],exp)\n        exp=exp#*scaler\n        node_features1=self.cell_encoder1(exp)\n        node_features2 = self.cell_encoder1(x["type_exp"])#*scaler)\n        node_features=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),node_features2,node_features1)\n\n        if self.use_cell_type_embedding:\n            node_features=node_features+self.cell_encoder2(x["cell_types"])\n\n        # calculate ligand scores\n        B,N,C=node_features.shape\n        data_ligand=[]\n        for i in range(len(self.ligands_info[0])):\n            data_ligandi=torch.ones((B,N),device=node_features.device)\n            for j in range(len(self.ligands_index[i])):\n                data_ligandi = data_ligandi * torch.mean(exp[:,:,self.ligands_index[i][j]],dim=-1)\n            data_ligandi=torch.pow(data_ligandi,exponent=1 / len(self.ligands_index[i]))\n            data_ligand.append(data_ligandi)\n        data_ligand=torch.stack(data_ligand,dim=-1)\n\n        # distance scaler\n        distance_scaler = self.distance_scaler(x["distance_matrix"])\n\n        # ligand score scaled by distance\n        ligand_score=distance_scaler*data_ligand\n        if ligand_feature_mask is not None:\n            ligand_score = ligand_score * ligand_feature_mask\n        edge_ligand=self.ligands_encoder(ligand_score)\n\n        # edge features\n        edge_distance=self.distance_encoder(x["distance_matrix"])\n        edges = self.edge_encoder(torch.concat([edge_distance,edge_ligand],dim=-1))*self.scaler\n\n        # distance embedding\n        embedding = self.distance_embedding(x["distance_matrix"])\n        return [node_features,edges.unsqueeze(dim=1),embedding.unsqueeze(dim=1)]\n\n\n\n', '__init__.py': '"""\nSpaGAT package.\n"""\n\nfrom .gene_program_model import SpaGP, SpaGP_Loss\nfrom .dataloader import SPAGAT_dataset\n\n__all__ = [\n    "SpaGP",\n    "SpaGP_Loss",\n    "SPAGAT_dataset",\n]', 'gene_program_model.py': '"""\nSpatial Gene Program Model (SpaGP)\n\nCore idea:\n    y_i = μ(cell_type_i) + Σ_k g_k(z_i) · p_k + ε\n\nwhere:\n    - μ(cell_type) is cell-type baseline expression (from type_exp)\n    - p_k ∈ R^genes is the k-th gene program (global, shared, orthogonal)\n    - g_k(z_i) ∈ R+ is scalar activation of program k for cell i (non-negative)\n    - z_i ∈ R^d is spatial context embedding from neighborhood via graph encoder\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .attention import GRIT_encoder\nfrom .embedding import Embedding, FFN\n\n\nclass GeneProgramHead(nn.Module):\n    """\n    The program-based prediction head.\n    \n    Given spatial context embedding z_i (from graph encoder),\n    predicts gene expression as:\n        y_i = baseline_i + Σ_k g_k(z_i) · p_k\n    """\n    def __init__(self, node_dim, n_genes, n_programs=16, n_neighbors=50,\n                 program_aware=False, message_decoder="free"):\n        super().__init__()\n        self.n_genes = n_genes\n        self.n_programs = n_programs\n        self.n_neighbors = n_neighbors\n        self.program_aware = program_aware\n        self.message_decoder = message_decoder\n        if message_decoder not in {"free", "program_aligned"}:\n            raise ValueError("message_decoder must be \'free\' or \'program_aligned\'.")\n        \n        # ===== Gene Programs p_k ∈ R^{K × genes} =====\n        # Initialized with Xavier, will be orthogonalized during training via loss penalty\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # ===== Context Aggregator: aggregate neighborhood into z_i =====\n        # Attention-weighted aggregation of neighbor node features → z_i\n        self.context_query = nn.Linear(node_dim, node_dim)\n        self.context_key = nn.Linear(node_dim, node_dim)\n        self.context_proj = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, node_dim)\n        )\n        self.scale = node_dim ** 0.5\n        \n        # ===== Activation Network: z_i → g(z_i) ∈ R+^K =====\n        self.activation_net = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, n_programs),\n            nn.Softplus()  # ensures g_k ≥ 0\n        )\n        \n        # ===== Baseline: cell-type expression → gene-level baseline =====\n        self.baseline_head = nn.Sequential(\n            nn.Linear(n_genes, n_genes),\n            nn.GELU(),\n            nn.Linear(n_genes, n_genes)\n        )\n\n        # Only program-aware SpaGP uses this decoder branch.  Keeping it absent\n        # from the legacy construction preserves existing checkpoint layouts.\n        if program_aware:\n            if message_decoder == "free":\n                self.message_to_genes = nn.Linear(node_dim, n_genes)\n            else:\n                # A shared scalar decoder yields q_k from each program message\n                # m_k.  Its gene effect is constrained to that program\'s P_k.\n                self.message_to_program = nn.Linear(node_dim, 1)\n    \n    def forward(self, node_features, type_exp_center, encoder_program_state=None, program_mask=None):\n        """\n        Args:\n            node_features: (B, N, node_dim) — encoder output for center + neighbors\n            type_exp_center: (B, n_genes) — cell-type average expression for center cell\n            encoder_program_state: Optional state returned by a program-aware\n                GRIT encoder.  When supplied, its center-cell activations replace\n                this head\'s legacy context/activation inference.\n            program_mask: Optional inference-only ``(K,)`` 0/1 mask. It is\n                applied only to program-specific decoder contributions; the\n                shared cell-type baseline remains unchanged.\n        \n        Returns:\n            y_pred: (B, n_genes)\n            program_info: dict with activations and programs for downstream analysis\n        """\n        B, N, C = node_features.shape\n        \n        if encoder_program_state is None:\n            # Legacy decoder path: retain the original head computation and\n            # parameter layout so existing SpaGP checkpoints remain usable.\n            center = node_features[:, 0:1, :]\n            neighbors = node_features[:, 1:, :]\n            Q = self.context_query(center)\n            K = self.context_key(neighbors)\n            attn = F.softmax((Q @ K.transpose(-1, -2)) / self.scale, dim=-1)\n            z_i = self.context_proj(attn @ neighbors).squeeze(1)\n            g = self.activation_net(z_i)\n            program_attention = None\n            program_messages = None\n        else:\n            # Program-aware decoder path: graph message passing has already\n            # produced non-negative activations for every node/program pair.\n            # The center-cell row is the activation vector used to decode genes.\n            g = encoder_program_state[\'activations\'][:, 0, :]\n            z_i = node_features[:, 0, :]\n            program_attention = encoder_program_state[\'attention\'][:, 0, 1:, :]\n            program_messages = encoder_program_state[\'messages\'][:, 0, :, :]\n\n            # Keep the historical (B, N-1) attn_weights field as an activation-\n            # weighted summary of the full program-specific attention tensor.\n            gate_weights = g / (torch.sum(g, dim=-1, keepdim=True) + 1e-8)\n            attn = torch.sum(program_attention * gate_weights.unsqueeze(dim=1), dim=-1).unsqueeze(dim=1)\n        \n        # --- Step 3: Normalize programs (unit norm per program) ---\n        programs_normalized = F.normalize(self.programs, dim=-1)  # (K, genes)\n        if program_mask is not None:\n            program_mask = torch.as_tensor(\n                program_mask, device=programs_normalized.device, dtype=programs_normalized.dtype\n            )\n            if program_mask.ndim != 1 or program_mask.shape[0] != self.n_programs:\n                raise ValueError(\n                    f"program_mask must have shape ({self.n_programs},); got {tuple(program_mask.shape)}."\n                )\n        mask_is_identity = program_mask is None or bool(torch.all(program_mask == 1))\n        \n        # --- Step 4: Program contribution ---\n        # Σ_k g_k(z_i) · p_k = g @ P\n        # Keep the default expression byte-for-byte on the existing path.  For\n        # an intervention, mask each program\'s gene-program contribution before\n        # summing across k; no re-normalization occurs.\n        masked_g = g if mask_is_identity else g * program_mask.unsqueeze(0)\n        program_output = masked_g @ programs_normalized   # (B, genes)\n\n        if encoder_program_state is None:\n            message_output = None\n            message_coefficients = None\n        else:\n            if not self.program_aware:\n                raise RuntimeError(\n                    "GeneProgramHead received encoder program state but was not "\n                    "constructed with program_aware=True."\n                )\n            if self.message_decoder == "free":\n                # Legacy program-aware ablation: a flexible shared gene decoder\n                # consumes the gate-weighted sum of program messages.\n                if mask_is_identity:\n                    fused_program_message = torch.sum(\n                        program_messages * gate_weights.unsqueeze(dim=-1), dim=1\n                    )\n                    message_output = self.message_to_genes(fused_program_message)\n                else:\n                    # Distribute the free decoder\'s shared bias by the original\n                    # program weights: sum_k w_k (W m_k + b) equals the legacy\n                    # W sum_k(w_k m_k) + b when every mask is one, while an\n                    # all-zero mask removes the complete program message path.\n                    message_weights = gate_weights * program_mask.unsqueeze(0)\n                    message_components = F.linear(\n                        program_messages, self.message_to_genes.weight, self.message_to_genes.bias\n                    )\n                    message_output = torch.sum(message_components * message_weights.unsqueeze(dim=-1), dim=1)\n                message_coefficients = None\n            else:\n                # Program-aligned decoder: q_k is derived from m_k, then its\n                # contribution is restricted to the corresponding gene program\n                # P_k rather than passing through an unrestricted gene decoder.\n                message_coefficients = self.message_to_program(program_messages).squeeze(dim=-1)\n                masked_coefficients = (\n                    message_coefficients if mask_is_identity\n                    else message_coefficients * program_mask.unsqueeze(0)\n                )\n                message_output = masked_coefficients @ programs_normalized\n        \n        # --- Step 5: Baseline from cell-type expression ---\n        baseline = self.baseline_head(type_exp_center)  # (B, genes)\n        \n        # --- Step 6: Final prediction ---\n        y_pred = baseline + program_output\n        if message_output is not None:\n            y_pred = y_pred + message_output\n        \n        program_info = {\n            \'activations\': g,                    # (B, K) — program activation per cell\n            \'programs\': programs_normalized,      # (K, genes) — gene program vectors\n            \'context\': z_i,                       # (B, C) — spatial context embedding\n            \'attn_weights\': attn.squeeze(1),      # (B, N-1) — neighbor attention weights\n            \'baseline\': baseline,                 # (B, genes) — cell-type baseline\n            \'program_output\': program_output      # (B, genes) — program contribution\n        }\n        if program_attention is not None:\n            program_info[\'program_attention\'] = program_attention\n            program_info[\'program_messages\'] = program_messages\n            program_info[\'program_message_output\'] = message_output\n            program_info[\'program_message_coefficients\'] = message_coefficients\n        if program_mask is not None:\n            program_info[\'program_mask\'] = program_mask\n        \n        return y_pred, program_info\n\n\nclass SpaGP(nn.Module):\n    """\n    Spatial Gene Program Model.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16, program_aware=False, program_token_dim=32,\n                 message_decoder="free", routing_mode="program"):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        self.program_aware = program_aware\n        self.program_token_dim = program_token_dim\n        self.message_decoder = message_decoder\n        self.routing_mode = routing_mode\n\n        if program_aware and n_layers < 1:\n            raise ValueError("SpaGP program-aware mode requires at least one GRIT encoder layer.")\n        \n        self.embeddings = Embedding(genes, ligands_info, node_dim, edge_dim,\n                                    use_cell_type_embedding=use_cell_type_embedding)\n        self.encoders = nn.ModuleList([\n            GRIT_encoder(node_dim, edge_dim, num_heads, att_dim, program_aware=program_aware, routing_mode=routing_mode)\n            for i in range(n_layers)  \n        ])\n\n        if program_aware:\n            # The learned bank is intentionally in a separate latent space P.\n            # Its shared projection supplies (K, edge_dim) tokens required by\n            # every GRIT layer, keeping program identity fixed across depth.\n            self.program_tokens = nn.Parameter(torch.empty(n_programs, program_token_dim))\n            nn.init.xavier_uniform_(self.program_tokens)\n            self.program_token_projection = nn.Linear(program_token_dim, edge_dim, bias=False)\n\n        self.program_head = GeneProgramHead(\n            node_dim=node_dim,\n            n_genes=len(genes),\n            n_programs=n_programs,\n            program_aware=program_aware,\n            message_decoder=message_decoder,\n        )\n    \n    def forward(self, x, program_mask=None, program_edge_mask=None,ligand_feature_mask=None):\n        """\n        Args:\n            x: dict from dataloader with keys:\n                "x": (B, N, genes), "type_exp": (B, N, genes),\n                "cell_types": (B, N), "position_x": (B, N), "position_y": (B, N),\n                "y": (B, genes)\n        \n        Returns:\n            y_pred: (B, genes)\n            program_info: dict with activations, programs, context, etc.\n        """\n        # Center cell\'s type expression (for baseline)\n        type_exp_center = x["type_exp"][:, 0, :]  # (B, genes)\n        encoded = self.embeddings(x,ligand_feature_mask=ligand_feature_mask)   # [node, edge, distance]\n        if self.program_aware:\n            program_tokens = self.program_token_projection(self.program_tokens)\n            final_program_state = None\n            for encoder in self.encoders:\n                # Program state is analysis metadata; only graph tensors are\n                # forwarded to the next GRIT layer.\n                encoded = encoder(\n                    encoded[:3], program_tokens=program_tokens, program_edge_mask=program_edge_mask\n                )\n                final_program_state = encoded[3]\n        else:\n            for encoder in self.encoders:\n                encoded = encoder(encoded)\n            final_program_state = None\n        node_features = encoded[0]     # (B, N, node_dim)\n        \n        # Program-based prediction\n        y_pred, program_info = self.program_head(\n            node_features, type_exp_center, encoder_program_state=final_program_state,\n            program_mask=program_mask,\n        )\n        \n        return y_pred, program_info\n\n\nclass SpaGP_Loss(nn.Module):\n    """\n    Loss function for SpaGP model:\n        L = L_mse + λ_orth * L_orthogonality + λ_sparse * L_sparsity\n    """\n    def __init__(self, gene_list, interaction_gene_list1, \n                 lambda_orth=0.1, lambda_sparse=0.01):\n        super().__init__()\n        interaction_gene_list = list(set(elem for sublist in interaction_gene_list1[0] for elem in sublist))\n        \n        self.interaction_gene_index = []\n        self.not_interaction_gene_index = []\n        for i in range(len(gene_list)):\n            if gene_list[i] in interaction_gene_list:\n                self.interaction_gene_index.append(i)\n            else:\n                self.not_interaction_gene_index.append(i)\n        self.interaction_gene_index = torch.LongTensor(self.interaction_gene_index)\n        self.not_interaction_gene_index = torch.LongTensor(self.not_interaction_gene_index)\n        \n        self.mse = nn.MSELoss()\n        self.lambda_orth = lambda_orth\n        self.lambda_sparse = lambda_sparse\n    \n    def orthogonality_loss(self, programs):\n        """\n        Soft orthogonality penalty: Σ_{i≠j} (p_i · p_j)²\n        \n        Args:\n            programs: (K, genes) — normalized program vectors\n        """\n        # Gram matrix: (K, K)\n        gram = programs @ programs.T  # p_i · p_j for all pairs\n        # Zero out diagonal (self-similarity = 1, we don\'t penalize that)\n        K = programs.shape[0]\n        if K < 2:\n            # A single program has no program pair, so orthogonality/diversity\n            # is defined exactly as zero rather than evaluating 0 / (K * (K-1)).\n            return programs.new_zeros(())\n        mask = 1.0 - torch.eye(K, device=programs.device)\n        off_diag = gram * mask\n        # Penalty: sum of squared off-diagonal entries\n        return torch.sum(off_diag ** 2) / (K * (K - 1))\n    \n    def sparsity_loss(self, activations):\n        """\n        L1 sparsity on program activations.\n        Encourages each cell to activate only a few programs.\n        \n        Args:\n            activations: (B, K) — non-negative program activations\n        """\n        return torch.mean(activations)  # L1 of non-negative = mean\n    \n    def forward(self, y_pred_tuple, y):\n        """\n        Args:\n            y_pred_tuple: (y_pred, program_info) from model\n            y: (B, genes) ground truth\n        \n        Returns:\n            total_loss: scalar (for backward)\n            mse_not_interact: scalar (for monitoring, detached)\n            orth_loss: scalar (for monitoring, detached)\n            sparse_loss: scalar (for monitoring, detached)\n        """\n        y_pred, program_info = y_pred_tuple\n        \n        # Main prediction loss\n        mse_loss = self.mse(y_pred, y)\n        \n        # Orthogonality penalty\n        orth_loss = self.orthogonality_loss(program_info[\'programs\'])\n        \n        # Sparsity penalty\n        sparse_loss = self.sparsity_loss(program_info[\'activations\'])\n        \n        # Total\n        total_loss = mse_loss + self.lambda_orth * orth_loss + self.lambda_sparse * sparse_loss\n        \n        # Monitoring: MSE on non-interaction genes\n        mse_not_interact = self.mse(\n            y_pred[:, self.not_interaction_gene_index],\n            y[:, self.not_interaction_gene_index]\n        ).detach().cpu()\n        \n        return (total_loss, mse_not_interact, \n                orth_loss.detach().cpu(), sparse_loss.detach().cpu())\nclass SpaGP_NoSpatial(nn.Module):\n    """\n    Ablation model: g_k predicted from center cell expression only.\n    No neighborhood information → if VE ≈ full model, spatial story fails.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        n_genes = len(genes)\n        \n        # Programs (same as SpaGP)\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # Activation from center cell expression ONLY (no spatial context)\n        self.activation_net = nn.Sequential(\n            nn.Linear(n_genes, 128),\n            nn.GELU(),\n            nn.Linear(128, n_programs),\n            nn.Softplus()\n        )\n    \n    def forward(self, x):\n        # Only use center cell\'s raw expression — NO neighbors\n        center_exp = x["type_exp"][:, 0, :] + x["x"][:, 0, :]  # (B, genes)\n        \n        g = self.activation_net(center_exp)  # (B, K)\n        programs_norm = F.normalize(self.programs, dim=-1)\n        y_pred = g @ programs_norm\n        \n        program_info = {\n            \'activations\': g,\n            \'programs\': programs_norm,\n            \'context\': center_exp,\n        }\n        return y_pred, program_info\n', 'dataloader.py': 'import torch\nfrom torch.utils.data import Dataset\nimport os\nimport numpy as np\nimport pandas as pd\n\nclass SPAGAT_dataset(Dataset):\n    def __init__(self, processed_dir, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n\n        samples = []\n        for filei in os.listdir(processed_dir):\n            if filei.find("_TypeExp.npz") >= 0:\n                samples.append(filei.split("_TypeExp.npz")[0])\n        self.samples = list(sorted(list(set(samples))))\n        print("Have samples:", len(self.samples), self.samples)\n\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                # Assign cell type numbers for each cell in the sample\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                # Only keep cell types present in both the dict and the global cell type dict\n                valid_cell_types = [ct for ct in dfi[col_cell_type].unique()\n                                   if ct in type_exp_dicti and ct in self.cell_types_dict]\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]\n                                       if cell_type in valid_cell_types], dim=0)\n                # Skip sample if no valid cell types\n                if len(valid_cell_types) == 0:\n                    print(f"Skip sample {samplei}: no valid cell types")\n                    continue\n                # Only keep rows with valid cell types\n                valid_mask = dfi[col_cell_type].isin(valid_cell_types)\n                dfi_valid = dfi[valid_mask].copy()\n                if len(dfi_valid) == 0:\n                    print(f"Skip sample {samplei}: no valid rows after filtering")\n                    continue\n                self.indexes.append(torch.LongTensor(dfi_valid.loc[:, self.index_index].values))\n                self.flags.append(dfi_valid.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi_valid.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi_valid.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi_valid.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi_valid.loc[:, "cell_type_number"].values))\n                # Rebuild type_exp for valid cell types\n                type_exp_valid = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi_valid[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp_valid)\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        # Find which sample this idx comes from\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        num_real_neighbors = len(indices)\n        neighbor_mask = torch.ones(num_real_neighbors)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "neighbor_mask": neighbor_mask\n        }\n\nclass SPAGAT_evaluate_dataset(Dataset):\n    def __init__(self, processed_dir, sample, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n        self.samples = [sample]\n        print("Have samples:", self.samples)\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp)\n                self.indexes.append(torch.LongTensor(dfi.loc[:, self.index_index].values))\n                self.flags.append(dfi.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi.loc[:, "cell_type_number"].values))\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        indices = torch.LongTensor(indices)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "NN": indices\n        }\n'}
SOURCE_HASHES = {name: hashlib.sha256(text.encode()).hexdigest() for name, text in FROZEN_SOURCE.items()}
code_root = Path('/content/SpaGAT_R3_5_7_frozen_source')
package = code_root/'spagat_r3_export'
package.mkdir(parents=True, exist_ok=True)
for name, text in FROZEN_SOURCE.items():
    (package/name).write_text(text)
if str(code_root) not in sys.path: sys.path.insert(0,str(code_root))
print('Using the project source returned in your locator ZIP; original Drive source is unchanged.')


## 3. 定义核对、逐基因分析和 masking 导出程序

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, gc, zipfile, traceback, platform
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

BASE = Path('/content/drive/MyDrive')
DATA = BASE/'GITIII-main/liver_workdir/gitiii/data/processed'
CANDIDATES = [BASE/'spagatv2/results/liver_free_k4_test/free_seed123_best.pth',
              BASE/'spagatv2/results/liver_free_k4_3epochs/free_seed123_best.pth']
REFERENCE = BASE/'spagatv2/analysis/liver_sender_origin_random_control/matched_sender_random_control_repeats.csv'
ACTIVATIONS = BASE/'spagatv2/analysis/programs/Liver_validation_program_activations.npz'
BATCH_SIZE = 32
REPEATS = 10
MASK_SEED = 123
RECEIVERS = ['tumor_1','tumor_2']

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for x in iter(lambda:f.read(1024*1024),b''): h.update(x)
    return h.hexdigest()

def jsonsave(path,obj):
    Path(path).write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str))

class DiskDataset:
    """Same sample/flag order and batch fields as the supplied loader; disk-backed expression."""
    def __init__(self,data,cache,genes):
        self.genes=list(genes);self.samples=sorted(p.name[:-12] for p in data.glob('*_TypeExp.npz'))
        if self.samples!=['CancerousLiver','NormalLiver']:
            raise ValueError(f'Unexpected samples: {self.samples}')
        self.meta=[]; alltypes=set()
        for sample in self.samples:
            p=data/(sample+'.csv');header=pd.read_csv(p,nrows=0).columns
            cols=['subclass','centerx','centery']+(['flag'] if 'flag' in header else [])
            m=pd.read_csv(p,usecols=cols)
            if m['subclass'].isna().any():raise ValueError('Missing cell annotations')
            if 'flag' not in m:m['flag']=True
            if not m['flag'].isin([0,1,False,True]).all():raise ValueError('Nonbinary flags')
            alltypes.update(m['subclass']);self.meta.append(m)
        self.names=sorted(alltypes);self.type_ids={x:i for i,x in enumerate(self.names)}
        self.parts=[];self.refs=[];self.global_types=[];self.global_sample=[];self.global_rows=[]
        for k,(sample,m) in enumerate(zip(self.samples,self.meta)):
            n=len(m);directory=cache/sample;directory.mkdir(parents=True)
            exp=np.lib.format.open_memmap(directory/'expression.npy',mode='w+',dtype='float32',shape=(n,len(genes)))
            ids=np.lib.format.open_memmap(directory/'neighbors.npy',mode='w+',dtype='int64',shape=(n,50))
            types=m['subclass'].map(self.type_ids).to_numpy(dtype=np.int64)
            with np.load(data/(sample+'_TypeExp.npz'),allow_pickle=False) as z:
                if set(m['subclass'])-set(z.files):raise ValueError('Missing baseline types; refusing silent row filtering')
                baselines=np.zeros((len(self.names),len(genes)),np.float32)
                for name in set(m['subclass']):
                    v=z[name]
                    if v.shape!=(len(genes),) or not np.isfinite(v).all():raise ValueError('Invalid baseline')
                    baselines[self.type_ids[name]]=v
            offset=0;nc=[f'index_{i}' for i in range(50)]
            for chunk in pd.read_csv(data/(sample+'.csv'),usecols=list(genes)+nc,chunksize=4000):
                a=chunk.loc[:,genes].to_numpy(dtype=np.float32)
                nb=chunk.loc[:,nc].to_numpy()
                if not np.isfinite(a).all() or not np.isfinite(nb).all():raise ValueError('Nonfinite input')
                if not np.equal(nb,np.floor(nb)).all():raise ValueError('Noninteger neighbor indices')
                nb=nb.astype(np.int64)
                if nb.min()<0 or nb.max()>=n:raise ValueError('Neighbor index out of range')
                if not np.array_equal(nb[:,0],np.arange(offset,offset+len(a))):raise ValueError('index_0 is not the receiver row')
                exp[offset:offset+len(a)]=a;ids[offset:offset+len(a)]=nb;offset+=len(a)
            if offset!=n:raise ValueError('Row count mismatch')
            exp.flush();ids.flush()
            rows=np.flatnonzero(m['flag'].to_numpy()!=0)
            self.parts.append(dict(exp=exp,ids=ids,types=types,baseline=baselines,
                                   x=m.centerx.to_numpy(np.float32),y=m.centery.to_numpy(np.float32)))
            self.refs.extend((k,int(r)) for r in rows)
            self.global_types.extend(types[rows]);self.global_sample.extend([sample]*len(rows));self.global_rows.extend(rows)
            print('Cached',sample,n,'rows;',len(rows),'eligible dataset entries',flush=True)
        self.global_types=np.asarray(self.global_types);self.global_rows=np.asarray(self.global_rows)
        del self.meta;gc.collect()
    def __len__(self):return len(self.refs)
    def __getitem__(self,i):
        k,row=self.refs[int(i)];p=self.parts[k];nb=p['ids'][row];types=p['types'][nb]
        exp=np.asarray(p['exp'][nb],dtype=np.float32)
        return dict(x=exp,type_exp=p['baseline'][types],y=exp[0],cell_types=types.copy(),
                    position_x=p['x'][nb],position_y=p['y'][nb],neighbor_mask=np.ones(50,np.float32))
    def counts(self,indices):
        result=[]
        t1,t2=[self.type_ids[x] for x in RECEIVERS]
        for i in indices:
            k,r=self.refs[int(i)];p=self.parts[k];v=p['types'][p['ids'][r,1:]]
            result.append((int((v==t1).sum()),int((v==t2).sum())))
        return np.asarray(result,dtype=np.int64)

def instantiate(checkpoint,genes,ligands):
    from spagat_r3_export.gene_program_model import SpaGP
    state=checkpoint['model'];cfg=checkpoint.get('model_config',checkpoint.get('config',{}))
    if not isinstance(cfg,dict):cfg=vars(cfg)
    if 'program_tokens' not in state:raise ValueError('Not the expected program-aware model')
    args={k:cfg.get(k,v) for k,v in dict(node_dim=256,edge_dim=48,num_heads=2,n_layers=1,att_dim=8).items()}
    args.update(n_programs=state['program_tokens'].shape[0],program_token_dim=state['program_tokens'].shape[1],
                program_aware=True,message_decoder='free',routing_mode='program')
    if cfg.get('message_decoder','free')!='free' or cfg.get('routing_mode','program')!='program':
        raise ValueError('Checkpoint configuration differs from the supplied evaluation script')
    model=SpaGP(genes=genes,ligands_info=ligands,**args)
    model.load_state_dict(state,strict=True)
    return model.cuda().eval(),args

def predict(model,dataset,indices,directory):
    directory.mkdir(parents=True,exist_ok=True);shape=(len(indices),len(dataset.genes))
    pred=np.lib.format.open_memmap(directory/'prediction.npy',mode='w+',dtype='float32',shape=shape)
    target=np.lib.format.open_memmap(directory/'target.npy',mode='w+',dtype='float32',shape=shape)
    offset=0
    with torch.inference_mode():
        for batch in DataLoader(Subset(dataset,indices.tolist()),batch_size=BATCH_SIZE,shuffle=False,num_workers=0):
            batch={k:v.cuda() for k,v in batch.items()};p,_=model(batch);n=len(p)
            if not torch.isfinite(p).all():raise ValueError('Nonfinite model predictions')
            pred[offset:offset+n]=p.cpu().numpy();target[offset:offset+n]=batch['y'].cpu().numpy();offset+=n
            if offset%3200==0:print('Predicted',offset,'/',len(indices),flush=True)
    pred.flush();target.flush();return pred,target

def mse_rows(pred,target,selection):
    total=0.;n=0
    idx=np.flatnonzero(selection)
    for block in np.array_split(idx,max(1,len(idx)//1000+1)):
        e=pred[block].astype(np.float64)-target[block];total+=np.square(e).sum();n+=e.size
    return total/n

def gene_metrics(pred,target,selection,genes):
    # Two-pass centered moments; report undefined PCC explicitly for constant genes.
    idx=np.flatnonzero(selection);g=len(genes);sx=np.zeros(g);sy=np.zeros(g);ss=np.zeros(g);zero=np.zeros(g)
    blocks=np.array_split(idx,max(1,len(idx)//1000+1))
    for b in blocks:
        x=np.asarray(pred[b],np.float64);y=np.asarray(target[b],np.float64)
        sx+=x.sum(0);sy+=y.sum(0);ss+=((x-y)**2).sum(0);zero+=(y*y).sum(0)
    mx=sx/len(idx);my=sy/len(idx);xx=np.zeros(g);yy=np.zeros(g);xy=np.zeros(g)
    for b in blocks:
        x=np.asarray(pred[b],np.float64)-mx;y=np.asarray(target[b],np.float64)-my
        xx+=(x*x).sum(0);yy+=(y*y).sum(0);xy+=(x*y).sum(0)
    denom=np.sqrt(xx*yy);pcc=np.full(g,np.nan);np.divide(xy,denom,out=pcc,where=denom>0)
    ev=np.full(g,np.nan);np.divide(ss,zero,out=ev,where=zero>0);ev=100*(1-ev)
    return pd.DataFrame(dict(gene=genes,n_cells=len(idx),PCC=pcc,MSE=ss/len(idx),MSE0=zero/len(idx),
                             EV_percent=ev,observed_mean=my,predicted_mean=mx,
                             observed_variance=yy/len(idx),predicted_variance=xx/len(idx),
                             PCC_defined=denom>0))

def mask_positions(types,ids,condition,rng):
    result=[]
    for row in types:
        a=np.flatnonzero(row[1:]==ids[0]);b=np.flatnonzero(row[1:]==ids[1]);m=min(len(a),len(b))
        if m==0:raise ValueError('Ineligible receiver reached masking')
        pool=a if condition=='tumor_1' else b if condition=='tumor_2' else np.arange(49)
        result.append(rng.choice(pool,size=m,replace=False)+1)
    return result

def export_masking(model,ds,all_indices,full,truth,types,counts,out,cache):
    rows=[];scalar=[];g=len(ds.genes);conditions=['tumor_1','tumor_2','random']
    ids=[ds.type_ids[x] for x in RECEIVERS];both=(counts>0).all(1)
    for receiver_no,receiver in enumerate(RECEIVERS):
        positions=np.flatnonzero((types==ds.type_ids[receiver])&both)
        indices=all_indices[positions];n=len(indices)
        for ci,condition in enumerate(conditions):
            avgpath=cache/f'{receiver}__{condition}__mean_delta.npy'
            avg=np.lib.format.open_memmap(avgpath,mode='w+',dtype='float32',shape=(n,g));avg[:]=0
            for repeat in range(REPEATS):
                rng=np.random.default_rng(MASK_SEED+repeat*1000+ci*100+receiver_no)
                signed=np.zeros(g);absolute=np.zeros(g);sq=np.zeros(g);offset=0
                with torch.inference_mode():
                    for batch in DataLoader(Subset(ds,indices.tolist()),batch_size=BATCH_SIZE,shuffle=False,num_workers=0):
                        choices=mask_positions(batch['cell_types'].numpy(),ids,condition,rng)
                        size=len(choices);mask=torch.ones((size,50,50,model.n_programs),dtype=torch.bool,device='cuda')
                        for r,chosen in enumerate(choices):mask[r,0,torch.as_tensor(chosen,device='cuda'),:]=False
                        b={k:v.cuda() for k,v in batch.items()};p,_=model(b,program_edge_mask=mask)
                        p=p.cpu().numpy();pos=positions[offset:offset+size]
                        if not np.isfinite(p).all():raise ValueError('Nonfinite masked predictions')
                        delta=p.astype(np.float64)-full[pos]
                        avg[offset:offset+size]+=np.asarray(delta/REPEATS,np.float32)
                        signed+=delta.sum(0);absolute+=np.abs(delta).sum(0)
                        sq+=((p.astype(np.float64)-truth[pos])**2).sum(0);offset+=size
                for k,gene in enumerate(ds.genes):
                    rows.append(dict(receiver=receiver,condition=condition,repeat=repeat,gene=gene,n_receivers=n,
                                     mean_signed_delta=signed[k]/n,mean_absolute_delta=absolute[k]/n,masked_MSE=sq[k]/n))
                fm=mse_rows(full,truth,(types==ds.type_ids[receiver])&both)
                scalar.append(dict(receiver=receiver,condition=condition,repeat=repeat,n_receivers=n,
                                   mean_edges_removed=float(np.min(counts[positions],axis=1).mean()),
                                   full_MSE=fm,masked_MSE=float(sq.sum()/(n*g)),
                                   mean_abs_prediction_change=float(absolute.sum()/(n*g)),
                                   relative_MSE_increase_percent=float(100*(sq.sum()/(n*g)/fm-1))))
                pd.DataFrame(scalar).to_csv(out/'masking_scalar_repeats.csv',index=False)
                print(receiver,condition,'repeat',repeat+1,'/',REPEATS,flush=True)
            avg.flush();del avg
            shutil.copy2(avgpath,out/'arrays'/avgpath.name)
            np.save(out/'arrays'/f'{receiver}__{condition}__dataset_indices.npy',indices)
    table=pd.DataFrame(rows);table.to_csv(out/'signed_masking_gene_repeats.csv',index=False)
    summary=table.groupby(['receiver','condition','gene'],sort=False).agg(
        n_receivers=('n_receivers','first'),mean_signed_delta=('mean_signed_delta','mean'),
        SD_signed_delta_across_repeats=('mean_signed_delta','std'),
        mean_absolute_delta=('mean_absolute_delta','mean'),
        SD_absolute_delta_across_repeats=('mean_absolute_delta','std'),
        mean_masked_MSE=('masked_MSE','mean')).reset_index()
    summary.to_csv(out/'signed_masking_gene_summary.csv',index=False)

def run_export():
    if not torch.cuda.is_available():raise RuntimeError('请选择 Colab GPU 后再运行；本程序不会改用 CPU 长时间推理。')
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
    out=BASE/'SpaGAT_revision'/('R3_5_7_gene_signed_'+stamp);out.mkdir(parents=True,exist_ok=False)
    cache=Path('/content')/('R3_5_7_cache_'+stamp);cache.mkdir(exist_ok=False)
    provenance=dict(status='running',new_training=False,new_inference=True,
                    scope='Existing-model validation gene recovery and matched-count signed masking; no independent liver test',
                    data_dir=str(DATA),batch_size=BATCH_SIZE,repeats=REPEATS,mask_seed=MASK_SEED,
                    torch_version=torch.__version__,numpy_version=np.__version__,python_version=platform.python_version(),
                    candidate_checks=[],source_hashes=SOURCE_HASHES,
                    program_recovery_status='Pending external biological gene sets; latent channels are not used as gene sets')
    try:
        torch.manual_seed(123);torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
        torch.backends.cudnn.benchmark=False
        genes=list(torch.load(DATA.parent/'genes.pth',map_location='cpu',weights_only=False))
        ligands=torch.load(DATA.parent/'ligands.pth',map_location='cpu',weights_only=False)
        if len(genes)!=1000 or len(set(genes))!=len(genes):raise ValueError('Unexpected or duplicated Liver genes')
        if shutil.disk_usage('/content').free<8*1024**3:raise RuntimeError('Local runtime needs at least 8 GB free disk space')
        ds=DiskDataset(DATA,cache/'data',genes)
        provenance['dataset_size']=len(ds);provenance['cell_type_names']=ds.names
        provenance['input_files']=[dict(path=str(p),size=p.stat().st_size,mtime=p.stat().st_mtime)
                                   for p in DATA.iterdir() if p.suffix in {'.csv','.npz'}]
        reference=pd.read_csv(REFERENCE);provenance['reference_sha256']=sha(REFERENCE)
        expected={}
        for name in RECEIVERS:
            x=reference[reference.receiver==name]
            if x.empty or x.n_receivers.nunique()!=1 or x.full_MSE.max()-x.full_MSE.min()>1e-7:
                raise ValueError('Inconsistent reference records')
            expected[name]=(int(x.n_receivers.iloc[0]),float(x.full_MSE.iloc[0]))
        selected=[]
        for ci,path in enumerate(CANDIDATES):
            rec=dict(checkpoint=str(path),matched=False)
            try:
                cp=torch.load(path,map_location='cpu',weights_only=False)
                val=np.asarray(cp['shared_split']['val_indices'],dtype=np.int64)
                train=np.asarray(cp['shared_split']['train_indices'],dtype=np.int64)
                split=json.loads((path.parent/'shared_split.json').read_text())
                if not np.array_equal(val,np.asarray(split['val_indices'])):raise ValueError('Saved split mismatch')
                if len(np.unique(val))!=len(val) or len(np.intersect1d(val,train)) or val.min()<0 or val.max()>=len(ds):
                    raise ValueError('Invalid validation indices')
                with np.load(ACTIVATIONS,allow_pickle=False) as old:
                    if not np.array_equal(val,old['val_indices']):raise ValueError('Validation order differs from saved activation export')
                    actual=np.asarray(ds.names)[ds.global_types[val]]
                    if not np.array_equal(actual,old['cell_type']):raise ValueError('Cell annotation mapping differs from saved export')
                    for coord,key in [('x','position_x'),('y','position_y')]:
                        coords=np.array([ds.parts[ds.refs[int(i)][0]][coord][ds.refs[int(i)][1]] for i in val])
                        if not np.allclose(coords,old[key],rtol=0,atol=1e-4):raise ValueError('Receiver coordinates differ from saved export')
                use=val[np.isin(ds.global_types[val],[ds.type_ids[x] for x in RECEIVERS])]
                types=ds.global_types[use];counts=ds.counts(use);both=(counts>0).all(1)
                rec['counts']={n:int(((types==ds.type_ids[n])&both).sum()) for n in RECEIVERS}
                if any(rec['counts'][n]!=expected[n][0] for n in RECEIVERS):raise ValueError('Eligible receiver count mismatch')
                model,args=instantiate(cp,genes,ligands);del cp;gc.collect()
                pred,target=predict(model,ds,use,cache/f'candidate_{ci}')
                values={n:mse_rows(pred,target,(types==ds.type_ids[n])&both) for n in RECEIVERS}
                rec.update(model_config=args,full_MSE=values,checkpoint_sha256=sha(path),
                           split_sha256=sha(path.parent/'shared_split.json'),n_validation=len(val))
                # Tolerance allows float32/reduction differences, not broad performance similarity.
                rec['matched']=all(np.isclose(values[n],expected[n][1],rtol=1e-5,atol=1e-6) for n in RECEIVERS)
                if rec['matched']:selected.append((path,ci,use,types,counts,args))
                del model,pred,target;torch.cuda.empty_cache();gc.collect()
            except Exception as e:rec['error']=str(e)
            provenance['candidate_checks'].append(rec);jsonsave(out/'provenance.json',provenance)
            print('Checkpoint check:',rec,flush=True)
        if len(selected)!=1:
            raise RuntimeError(f'Need exactly one matching checkpoint; found {len(selected)}. Export stopped; upload the diagnostic ZIP.')
        path,ci,use,types,counts,args=selected[0]
        provenance['selected_checkpoint']=str(path)
        provenance['identity_evidence']='Unique agreement among two candidates on saved validation order, cell labels, coordinates, eligible counts and two full MSE values; not a recovered historical command.'
        full=np.load(cache/f'candidate_{ci}/prediction.npy',mmap_mode='r');truth=np.load(cache/f'candidate_{ci}/target.npy',mmap_mode='r')
        # The supplied loader returns processed expression as y, without extra centering.
        # Record sign/centering diagnostics instead of silently subtracting a second baseline.
        provenance['target_semantics']='Target is the processed CSV gene vector used by the supplied loader; no additional baseline subtraction. Residual interpretation still requires preprocessing provenance.'
        (out/'arrays').mkdir();shutil.copy2(cache/f'candidate_{ci}/prediction.npy',out/'arrays/prediction.npy')
        shutil.copy2(cache/f'candidate_{ci}/target.npy',out/'arrays/target.npy')
        np.save(out/'arrays/dataset_indices.npy',use)
        (out/'genes.json').write_text(json.dumps(genes))
        pd.DataFrame(dict(dataset_index=use,sample=[ds.global_sample[i] for i in use],
                          csv_row=ds.global_rows[use],cell_type=np.asarray(ds.names)[types],
                          n_tumor1_neighbors=counts[:,0],n_tumor2_neighbors=counts[:,1],
                          masking_eligible=(counts>0).all(1))).to_csv(out/'receiver_metadata.csv',index=False)
        for n in RECEIVERS:
            gene_metrics(full,truth,types==ds.type_ids[n],genes).to_csv(out/f'{n}_gene_recovery.csv',index=False)
        model,_=instantiate(torch.load(path,map_location='cpu',weights_only=False),genes,ligands)
        export_masking(model,ds,use,full,truth,types,counts,out,cache)
        provenance['status']='complete_gene_and_matched_count_masking'
        print('基因预测和等数量 signed masking 导出完成；biological program analysis 尚未完成。')
    except Exception as e:
        provenance['status']='stopped';provenance['error']=str(e)
        (out/'error_trace.txt').write_text(traceback.format_exc());print('停止：',str(e))
    finally:
        jsonsave(out/'provenance.json',provenance)
        (out/'README.txt').write_text('Small ZIP contains tables and provenance, not large arrays.\n'
          'Arrays remain in the Drive output directory. prediction/target rows align with receiver_metadata.csv; columns align with genes.json.\n'
          'Signed delta is masked prediction minus full prediction. Saved mean_delta arrays average cellwise signed delta over 10 repeats; they are not individual-repeat predictions.\n'
          'Repeat SD describes fixed-model neighbor sampling only. Gene PCC is NaN for zero-variance observed/predicted genes; no genes are removed.\n'
          'Matched-count masking only; this is not a rerun of the fraction50 control. No biological gene sets or disease identities have been inferred.\n'
          'Do not interpret outputs as residual recovery until processed-target semantics are verified; no second residualization was applied.\n')
        archive=out.with_suffix('.zip')
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
            for p in out.iterdir():
                if p.is_file():z.write(p,out.name+'/'+p.name)
        print('请上传：',archive,flush=True)
    return str(archive)


## 4. 运行并下载结果 ZIP

GPU 推理可能持续较长时间，请保持运行时连接。每个阶段会打印进度。结果保存在 Drive 的新目录 `SpaGAT_revision/R3_5_7_gene_signed_时间戳/`。

完成或检查失败均会生成 ZIP，**将 ZIP 上传回当前对话即可**。ZIP 不含大型矩阵，矩阵保留在该目录的 `arrays/`。如果 GPU 显存不足，改上方 `BATCH_SIZE = 32` 为 `16`，重新运行第 3、4 步；不要更改匹配阈值强行通过。


In [ ]:
archive_path = run_export()
from google.colab import files
files.download(archive_path)


## 输出说明

- `provenance.json`：核对是否成功、唯一匹配 checkpoint、配置、参考文件哈希和限制。
- `tumor_1_gene_recovery.csv` / `tumor_2_gene_recovery.csv`：各群体全部 validation receivers 的逐基因 PCC、MSE、MSE0、EV、均值和方差。恒定基因 PCC 标为缺失，保留该基因。
- `signed_masking_gene_repeats.csv` / `signed_masking_gene_summary.csv`：两类 eligible receivers、三种条件（tumor_1/tumor_2/random）、10 次重复的逐基因有符号变化和绝对变化。SD 仅代表固定模型下的抽样变异。
- `masking_scalar_repeats.csv`：用于对照旧结果的整体指标。
- `receiver_metadata.csv`、`genes.json`、`arrays/`：预测行列的对应关系。`mean_delta` 数组保存同一 receiver 的 10 次 signed delta 均值，不是各次 masked prediction 原始矩阵；各次逐基因汇总另存 CSV。

差值始终定义为 **masked prediction − full prediction**。正负表示模型预测如何变化，不等同于移除细胞后真实生物学表达升降。本 notebook 不自动命名疾病状态、不做显著性检验、不声称 biological program 分析已完成。
